In [68]:
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F

print("🚀 Batch Scoring — Trip Duration")

catalog = "iceberg"

# ------------------------------------------------------------------
# 1️⃣ Charger features
# ------------------------------------------------------------------
features = spark.table(f"{catalog}.gold.ml_trip_features")

# ------------------------------------------------------------------
# 2️⃣ Charger modèle RF
# ------------------------------------------------------------------
model_path = "/models/rf_trip_duration"
rf_model = PipelineModel.load(model_path)

# ------------------------------------------------------------------
# 3️⃣ Scoring
# ------------------------------------------------------------------
predictions = rf_model.transform(features)

scored = (
    predictions
    .withColumnRenamed("prediction", "predicted_duration")
    .withColumn(
        "prediction_error",
        F.col("predicted_duration") - F.col("trip_duration_minutes")
    )
    .withColumn(
        "abs_error",
        F.abs("prediction_error")
    )
    .withColumn(
        "is_peak_hour",
        F.col("hour_of_day").between(7, 9) | F.col("hour_of_day").between(16, 19)
    )
    .withColumn("model_version", F.lit("rf_v1"))
)

# ------------------------------------------------------------------
# 4️⃣ Sélection finale
# ------------------------------------------------------------------
final_df = scored.select(
    "tpep_pickup_datetime",
    "pickup_zone",
    "pickup_borough",
    "dropoff_zone",
    "trip_duration_minutes",
    "predicted_duration",
    "prediction_error",
    "abs_error",
    "is_peak_hour",
    "model_version"
)

# ------------------------------------------------------------------
# 5️⃣ Écriture GOLD
# ------------------------------------------------------------------
final_df.writeTo(f"{catalog}.gold.trip_duration_predictions") \
    .partitionedBy("model_version") \
    .createOrReplace()

print("✅ gold.trip_duration_predictions créée")


🚀 Batch Scoring — Trip Duration


✅ gold.trip_duration_predictions créée
